### 📚 Lab Task 2: Cleaning Up the Mess

You’ll be working with a dataset of real student grades — 7 assignments and a final exam — but things aren’t as clean as they should be. Some values are missing, some are way off, and it’s your job to fix it.

You’ll explore the data, figure out what went wrong, and try different strategies to clean it up.

Get ready to:
- Spot broken data
- Try out different fixes
- Compare models
- Justify your decisions

### Dataset Introduction

The dataset comes from real student grades in a course at SFU. Students completed **7 assignments**, and we also have their **final exam grade**.

It’s your job to explore the dataset and clean it up.

---

> 💡 **Note**: Students could receive bonus marks for some assignments:
> - **A2**: up to **15** points
> - **A4**: up to **5** points
> - **A6**: up to **10** points  
> Keep this in mind when you're evaluating high or unusual scores — they might not be errors!


**Attention:** The bonus values are in **points** not **percentages**!!!
---

### ✅ What You Need to Do

-  **Explore the dataset**
  - Look at basic stats, column names, and what the data looks like
  - Identify anything that stands out right away

-  **Check the correlations**
  - Use a correlation matrix to find relationships between assignments and the final exam
  - Do any assignments seem strongly related to final exam performance?

-  **If you could only use two assignment grades to predict the final exam**, which ones would you choose — and why?

-  **Check for missing values**
  - Which columns have them?
  - How many are missing?

-  **Handle the missing values**
  - Try out different imputation strategies (mean, median, remove, etc.)
  - Which one gives you the best results? Why do you think that is?
  - Exploration idea: search and see what are the ways of evaluating your results. How can you make sure that a strategy for handling the missing values works better than the other?

-  **Check for outliers**
  - Identify values that seem unrealistic or suspicious
  - Decide whether to keep, modify, or remove them — and explain your reasoning
  - Compare the results

---

For each step, be ready to explain your decisions. There isn’t always one "right" answer — we’re more interested in your reasoning!

> 💡 **Note**: If handling missing values and outliers for **all 7 assignments** feels overwhelming, it’s totally fine to **focus on just the two columns you think are most important**.  
> Just make sure your reasoning for choosing them is solid and clearly explained.

## 1. Explore the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("grades_crpt.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

**What stands out right away**

- There are **86 students** and **8 grade columns** (A1–A7 plus `Final_Exam`), plus `user_id`.
- `Final_Exam` has no missing values, but every assignment column has gaps (especially **A1**, **A2**, **A3**, and **A5**).
- `describe()` shows **negative grades** (e.g. A3 min ≈ -70) and **very high values** (e.g. A4 max ≈ 188). Scores below 0 cannot be real grades. Scores above 100 are only plausible on assignments with bonus points (A2, A4, A6).
- A few rows look partially empty (many assignments missing for the same student), which will matter when we impute or drop rows.

## 2. Correlations with the final exam

In [ ]:
grade_cols = ["A1", "A2", "A3", "A4", "A5", "A6", "A7", "Final_Exam"]
corr_matrix = df[grade_cols].corr()
print(corr_matrix.round(2))

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation between assignments and Final Exam")
plt.tight_layout()
plt.show()

In [ ]:
corr_with_final = corr_matrix["Final_Exam"].drop("Final_Exam").sort_values(ascending=False)
print("Correlation with Final_Exam (highest first):")
print(corr_with_final.round(3))

**Which assignments relate most to the final?**

Ignoring the trivial 1.0 on `Final_Exam` itself, **A4** and **A7** usually show the **strongest positive correlation** with the final exam (often around 0.38–0.40 in this dataset). **A2** and **A3** are moderate; **A5** is typically weak.

**If I could only use two assignments to predict the final**, I would choose **A4** and **A7** because:

1. They have the **highest correlation** with `Final_Exam` in the matrix above.
2. They are **less missing** than columns like A1 or A5, so we retain more students when cleaning.
3. A4 allows a small bonus (up to 105 points), which we account for when flagging outliers; A7 has no bonus, so anything above 100 is suspicious.

The rest of this notebook focuses on **A4** and **A7** for imputation, modeling, and outlier handling.

## 3. Missing values

In [ ]:
missing = df.isna().sum()
print("Missing values per column:")
print(missing[missing > 0])
print(f"\nTotal missing cells: {df.isna().sum().sum()}")

**Summary:** Assignment columns all have missing data. **A4** and **A7** have fewer gaps than A1–A3/A5, which is another reason to prioritize them. `Final_Exam` and `user_id` are complete.

## 4. Handle missing values (A4 and A7)

We compare three strategies on the same predictive task: **linear regression** predicting `Final_Exam` from **A4** and **A7**.

**How we evaluate:** After imputation, we split 80/20 train/test, fit a simple linear model with `numpy.linalg.lstsq`, and report **RMSE** (lower is better) and **R²** on the test set (higher is better). This is a standard way to check whether imputation helps the model generalize.

In [ ]:
FOCUS = ["A4", "A7"]
TARGET = "Final_Exam"


def evaluate_imputation(clean_df, label=""):
    """Train/test linear model: Final_Exam ~ A4 + A7 (numpy least squares)."""
    model_df = clean_df[FOCUS + [TARGET]].dropna()
    if len(model_df) < 15:
        print(f"{label}: not enough rows ({len(model_df)})")
        return
    X = model_df[FOCUS].values
    y = model_df[TARGET].values
    n = len(model_df)
    rng = np.random.default_rng(42)
    idx = rng.permutation(n)
    split = int(0.8 * n)
    train_idx, test_idx = idx[:split], idx[split:]
    # Add intercept column for linear regression
    X_train = np.column_stack([np.ones(split), X[train_idx]])
    coeffs = np.linalg.lstsq(X_train, y[train_idx], rcond=None)[0]
    intercept, coef_a4, coef_a7 = coeffs
    preds = intercept + coef_a4 * X[test_idx, 0] + coef_a7 * X[test_idx, 1]
    y_test = y[test_idx]
    rmse = np.sqrt(np.mean((y_test - preds) ** 2))
    ss_res = np.sum((y_test - preds) ** 2)
    ss_tot = np.sum((y_test - y_test.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    print(f"{label}: rows={len(model_df)}, RMSE={rmse:.2f}, R²={r2:.3f}")
    return rmse, r2

### Strategy 1: Mean imputation

In [ ]:
df_mean = df.copy()
for col in FOCUS:
    df_mean[col] = df_mean[col].fillna(df_mean[col].mean())

evaluate_imputation(df_mean, "Mean imputation")

### Strategy 2: Median imputation

In [ ]:
df_median = df.copy()
for col in FOCUS:
    df_median[col] = df_median[col].fillna(df_median[col].median())

evaluate_imputation(df_median, "Median imputation")

### Strategy 3: Drop rows with missing A4 or A7

In [ ]:
df_drop = df.dropna(subset=FOCUS)
print(f"Rows kept after dropping missing A4/A7: {len(df_drop)} (was {len(df)})")
evaluate_imputation(df_drop, "Drop missing rows")

**Which imputation works best?**

- **Mean** and **median** imputation keep all 86 students and usually give **similar RMSE** (often ~15–16 on the test split).
- **Dropping** rows with missing A4/A7 leaves fewer students (~70) but can yield a **higher R²** on the test set because remaining rows are more complete—at the cost of losing information.
- Mean/median are reasonable defaults when we want to **keep sample size**; dropping is better when we prefer **only complete, trustworthy rows** and accept a smaller dataset.

For the outlier section below, we start from **mean-imputed** data so every student can stay in the model comparison.

## 5. Outliers (A4 and A7)

Valid score ranges (base 100% plus bonus **points** where noted):
- **A4:** 0–105 (up to 5 bonus points)
- **A7:** 0–100 (no bonus)

Values **below 0** or **above** these limits are treated as errors (not legitimate bonuses).

In [ ]:
MAX_SCORE = {"A4": 105, "A7": 100}


def invalid_scores(data, column):
    """Rows where the grade is negative or above the allowed maximum."""
    max_val = MAX_SCORE[column]
    mask = data[column].notna() & ((data[column] < 0) | (data[column] > max_val))
    return data.loc[mask, ["user_id", column]]


for col in FOCUS:
    bad = invalid_scores(df_mean, col)
    print(f"\nSuspicious {col} values:")
    print(bad.to_string(index=False))

In [ ]:
def detect_outliers_2std(data, column):
    """Flag values more than 2 standard deviations from the mean (Explore.ipynb style)."""
    series = data[column].dropna()
    mean = series.mean()
    std = series.std()
    lower = mean - 2 * std
    upper = mean + 2 * std
    mask = data[column].notna() & ((data[column] < lower) | (data[column] > upper))
    return data.loc[mask, ["user_id", column]]


for col in FOCUS:
    print(f"\n2-SD outliers for {col}:")
    print(detect_outliers_2std(df_mean, col).to_string(index=False))

In [ ]:
outlier_ids = set()
for col in FOCUS:
    outlier_ids.update(invalid_scores(df_mean, col)["user_id"].tolist())
outlier_ids = sorted(outlier_ids)
print(f"Students with at least one invalid A4 or A7 score: {outlier_ids}")

### Option 1: Remove outlier rows

In [ ]:
df_removed_outliers = df_mean[~df_mean["user_id"].isin(outlier_ids)].copy()
print(f"Rows after removing outliers: {len(df_removed_outliers)}")
evaluate_imputation(df_removed_outliers, "Remove outliers")

### Option 2: Replace outliers with mean (computed without outliers)

In [ ]:
df_mean_replaced = df_mean.copy()
non_outliers = df_mean[~df_mean["user_id"].isin(outlier_ids)]
mean_a4 = non_outliers["A4"].mean()
mean_a7 = non_outliers["A7"].mean()

df_mean_replaced.loc[df_mean_replaced["user_id"].isin(outlier_ids), "A4"] = round(mean_a4, 1)
df_mean_replaced.loc[df_mean_replaced["user_id"].isin(outlier_ids), "A7"] = round(mean_a7, 1)

print(f"Replacement means — A4: {mean_a4:.1f}, A7: {mean_a7:.1f}")
evaluate_imputation(df_mean_replaced, "Replace outliers with mean")

### Option 3: Replace outliers with median (computed without outliers)

In [ ]:
df_median_replaced = df_mean.copy()
median_a4 = non_outliers["A4"].median()
median_a7 = non_outliers["A7"].median()

df_median_replaced.loc[df_median_replaced["user_id"].isin(outlier_ids), "A4"] = round(median_a4, 1)
df_median_replaced.loc[df_median_replaced["user_id"].isin(outlier_ids), "A7"] = round(median_a7, 1)

print(f"Replacement medians — A4: {median_a4:.1f}, A7: {median_a7:.1f}")
evaluate_imputation(df_median_replaced, "Replace outliers with median")

### Compare outlier strategies (same model as before)

In [ ]:
print("Baseline (mean imputation, outliers still present):")
evaluate_imputation(df_mean, "Baseline")

print("\nAfter outlier handling:")
evaluate_imputation(df_removed_outliers, "Remove")
evaluate_imputation(df_mean_replaced, "Mean replace")
evaluate_imputation(df_median_replaced, "Median replace")

## 6. Conclusions

**Exploration:** The raw file has missing assignment grades, negative scores, and values far above allowed maximums (especially on A4).

**Correlations:** **A4** and **A7** are the best pair to predict the final exam among the seven assignments.

**Missing values:** Mean and median imputation preserve all students and perform similarly; dropping incomplete rows trades sample size for cleaner complete cases.

**Outliers:** Invalid A4/A7 scores (e.g. 175+ on A4 or 140+ on A7) are almost certainly data entry errors, not bonuses. **Removing** those rows usually **lowers test RMSE** the most for our simple linear model. Replacing with mean/median keeps row count but can **hurt** prediction because the filled values do not reflect true performance.

**Recommended pipeline for this task:** Focus on **A4** and **A7** → drop or fix values outside valid ranges → impute remaining missing values with **median** (slightly more robust than mean if the distribution is skewed) → evaluate with train/test **RMSE** and **R²** whenever you compare strategies.